# 03 — Federated K-Means Runs

Run federated k-means via FeatureCloud and aggregate results.

**What this notebook does:**
1. Prepare per-site input directories for the FeatureCloud fc_kmeans app.
2. Launch federated k-means tests (requires Docker + FeatureCloud CLI).
3. OR aggregate existing federated outputs (if FeatureCloud is unavailable).
4. Join federated cluster assignments with metadata.

**Prerequisites:**
- Run notebook 01 first (data preparation).
- For live runs: Docker running + FeatureCloud CLI installed.
- For aggregation only: existing zip results in `real_datasets/<dataset>/inputs/*/tests/`.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    discover_clients,
    load_feature_matrix,
    load_metadata,
    prepare_variant_inputs,
    aggregate_fed_clusters,
)
from evaluation_utils.featurecloud_kmeans_utils import (
    ensure_app_image,
    run_single_federated_variant,
    aggregate_existing_federated_output,
)

## Configuration

In [2]:
DATASETS = ["proteomics", "microarray", "proteomics_multibatch", "ccRCC_proteomics"]

# Set to True to launch a real FeatureCloud test.
# Set to False to only aggregate existing zip results.
RERUN_FEDERATED = True

# FeatureCloud settings (only needed when RERUN_FEDERATED=True)
APP_IMAGE = "fc_kmeans_upd"
APP_SOURCE_DIR = NOTEBOOK_DIR.parent / "federated_kmeans_upd"
CONTROLLER_HOST = "http://localhost:8000"
QUERY_INTERVAL = 5
TIMEOUT = 1800

OUTPUT_ROOT = NOTEBOOK_DIR

## Prepare FeatureCloud Inputs

Build per-site input directories with `intensities.tsv`, `design.tsv`,
and `config_kmeans.yml` — the format required by the fc_kmeans app.

This step creates `real_datasets/<dataset>/inputs/{before,corrected}/<site>/`
from the prepared matrices saved by notebook 01. It always runs (fast and idempotent)
so that the federated step below has up-to-date inputs.

In [3]:
# Always generate per-site input directories (fast, idempotent).
# These are needed for both fresh federated runs and for aggregating existing results.
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    prepared_dir = ds_dir / "prepared"

    # Load prepared data from notebook 01
    before = load_feature_matrix(prepared_dir / "before_matrix.tsv")
    corrected = load_feature_matrix(prepared_dir / "corrected_matrix.tsv")
    metadata = pd.read_csv(prepared_dir / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: preparing FC inputs for k={k_values}")
    print(f"{'='*60}")

    # Generate per-site input directories for the fc_kmeans app
    before_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="before",
        matrix=before, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    corrected_dir = prepare_variant_inputs(
        dataset_root=ds_dir, variant_name="corrected",
        matrix=corrected, metadata=metadata,
        clients=clients, k_values=k_values,
    )
    print(f"Created: {before_dir}")
    print(f"Created: {corrected_dir}")


proteomics: preparing FC inputs for k=[2, 5]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/corrected

microarray: preparing FC inputs for k=[2, 6]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/microarray/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/microarray/inputs/corrected

proteomics_multibatch: preparing FC inputs for k=[4]
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics_multibatch/inputs/before
Created: /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics_multibatch/inputs/corrected

ccRCC_proteomics: prepar

## Run or Aggregate Federated K-Means

When `RERUN_FEDERATED=True`, this launches a FeatureCloud test for each variant.
When `False`, it aggregates existing zip results from previous runs.

In [4]:
for ds_name in DATASETS:
    cfg = configs[ds_name]
    ds_dir = OUTPUT_ROOT / ds_name
    metadata = pd.read_csv(ds_dir / "prepared" / "metadata.tsv", sep="\t")
    clients = discover_clients(cfg)
    client_names = [c.name for c in clients]

    k_condition = int(metadata['condition'].nunique())
    k_batch = int(metadata['lab'].nunique())
    k_values = sorted(set([k_condition, k_batch]))

    print(f"\n{'='*60}")
    print(f"{ds_name}: {'running' if RERUN_FEDERATED else 'aggregating'} federated k-means")
    print(f"{'='*60}")

    for variant in ["before", "corrected"]:
        variant_input_dir = ds_dir / "inputs" / variant

        try:
            output_path = run_single_federated_variant(
                dataset_name=ds_name,
                variant_label=variant,
                variant_input_dir=variant_input_dir,
                dataset_root=ds_dir,
                metadata=metadata,
                client_names=client_names,
                k_values=k_values,
                app_image=APP_IMAGE,
                controller_host=CONTROLLER_HOST,
                query_interval=QUERY_INTERVAL,
                timeout=TIMEOUT,
                keep_extracted=False,
                aggregate_only=not RERUN_FEDERATED,
            )
            print(f"  {variant}: saved to {output_path}")
        except FileNotFoundError as e:
            print(f"  {variant}: SKIPPED — {e}")
        except RuntimeError as e:
            print(f"  {variant}: FAILED — {e}")


proteomics: running federated k-means
[proteomics] Starting FeatureCloud controller for 'before' data
Killing leftover containers...
Starting the FeatureCloud controller with the data directory /home/yuliya-cosybio/repos/cosybio/fedRBE/evaluation_clusterization_after_correction/real_datasets/proteomics/inputs/before


Downloading...: 3it [00:00, 4684.63it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_845959303', 'containerID': '76c988576ca02204', 'coordinator': False, 'frontendUrl': '/app-traffic/1108c8e2b9/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_618632282', 'containerID': 'eaec012b48d36584', 'coordinator': False, 'frontendUrl': '/app-traffic/6d756e9dcf/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_401301027', 'containerID': '991188a4f8435ca9', 'coordinator': False, 'frontendUrl': '/app-traffic/87af4b922f/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_167639549', 'containerID': '1135a02bd5285768', 'coordinator': False, 'frontendUrl': '/app-traffic/3f11f9380a/',

Downloading...: 3it [00:00, 1582.76it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_421926140', 'containerID': '0a7228814a6f4e5b', 'coordinator': True, 'frontendUrl': '/app-traffic/6c0baa18da/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_285268255', 'containerID': '246da00278c11541', 'coordinator': False, 'frontendUrl': '/app-traffic/bee16c1d22/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_126148335', 'containerID': 'b46a75bac7025931', 'coordinator': False, 'frontendUrl': '/app-traffic/bdd7354022/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_972178291', 'containerID': '1d737c73bde8455d', 'coordinator': False, 'frontendUrl': '/app-traffic/6cd7d1090a/

...

Controller started successfully, waiting 5s to ensure the controller is properly running!
[proteomics_multibatch] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_170087053', 'containerID': '403fad7f63ebbdc0', 'coordinator': False, 'frontendUrl': '/app-traffic/c040418e84/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_34104409', 'containerID': '2f36502b666c34c1', 'coordinator': False, 'frontendUrl': '/app-traffic/41b9a2909c/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_901056670', 'containerID': '7c9bed33d51ada69', 'coordinator': True, 'frontendUrl': '/app-traffic/4728814575/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 3, 'name': 'fc_fckmeansupd_775112718', 'containerID': '9083c636903ecad7', 'coordinator': False, 'frontendUrl': '/app-traffic/8

Downloading...: 3it [00:00, 1435.26it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'before' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_123807083', 'containerID': '21ec8969b80a5f13', 'coordinator': False, 'frontendUrl': '/app-traffic/d9589efb0d/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_952030450', 'containerID': '4b5bb90bf8436c34', 'coordinator': True, 'frontendUrl': '/app-traffic/45d84e3b6e/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_786258125', 'containerID': '327e73b1e1ba1652', 'coordinator': False, 'frontendUrl': '/app-traffic/c4cb799fa8/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio/re

Downloading...: 3it [00:00, 2045.00it/s]


Controller started successfully, waiting 5s to ensure the controller is properly running!
[ccRCC_proteomics] Running federated k-means for 'corrected' data
_______________EXPERIMENT_______________
Test started successfully!
Test finished successfully!
instances: [{'id': 0, 'name': 'fc_fckmeansupd_155532888', 'containerID': '03baa4e337881822', 'coordinator': False, 'frontendUrl': '/app-traffic/86b270b157/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 1, 'name': 'fc_fckmeansupd_968609113', 'containerID': 'e94d2fb88cdebbd0', 'coordinator': True, 'frontendUrl': '/app-traffic/7964c38e62/', 'message': 'terminal', 'state': None, 'progress': 1}, {'id': 2, 'name': 'fc_fckmeansupd_983195360', 'containerID': '119dc9c6ecfb0613', 'coordinator': False, 'frontendUrl': '/app-traffic/b0d1fe77ac/', 'message': 'terminal', 'state': None, 'progress': 1}]
TEST DONE:
_______________EXPERIMENT FINISHED SUCCESSFULLY_______________
[ccRCC_proteomics] Federated output saved: /home/yuliya-cosybio

## Verify Outputs

Check that federated metadata files were produced.

In [5]:
for ds_name in DATASETS:
    run_dir = OUTPUT_ROOT / ds_name / "kmeans_res" / "runs"
    for fname in ["1_metadata_before_fedclusters.tsv", "1_metadata_after_fedclusters.tsv"]:
        p = run_dir / fname
        if p.exists():
            df = pd.read_csv(p, sep="\t")
            print(f"{ds_name}/{fname}: {df.shape[0]} samples, cols={list(df.columns)}")
        else:
            print(f"{ds_name}/{fname}: NOT FOUND")

proteomics/1_metadata_before_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
proteomics/1_metadata_after_fedclusters.tsv: 118 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_5clusters']
microarray/1_metadata_before_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
microarray/1_metadata_after_fedclusters.tsv: 332 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_6clusters']
proteomics_multibatch/1_metadata_before_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
proteomics_multibatch/1_metadata_after_fedclusters.tsv: 72 samples, cols=['file', 'condition', 'lab', 'Fed_4clusters']
ccRCC_proteomics/1_metadata_before_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3clusters']
ccRCC_proteomics/1_metadata_after_fedclusters.tsv: 887 samples, cols=['file', 'condition', 'lab', 'Fed_2clusters', 'Fed_3c